# Etapa 2 — Extração e seleção de atributos

Continuação do `01_analise_exploratoria.ipynb` (etapa 1, análise exploratória). Este notebook parte dos dados brutos e produz o conjunto de atributos usado nas etapas de treinamento e avaliação.

Roteiro:
1. Configuração e carregamento
2. Construção da matriz de atributos (codificação das qualitativas e tratamento dos faltantes)
3. Separação treino/teste
4. Seleção de atributos (filtro, Lasso, estabilidade, comparação com Random Forest e conjunto final)

Toda a lógica está em `src/features.py` e `src/preprocessing.py`; o notebook só orquestra e mostra os resultados.

## 1. Configuração e carregamento

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

# Add the repository root to sys.path so that `src` can be imported
BASE_DIR = Path.cwd().resolve().parent
if str(BASE_DIR) not in sys.path:
    sys.path.append(str(BASE_DIR))

from src import utils, preprocessing, features

utils.setup_plotting()

In [ ]:
patients, drugs, data_dictionary = preprocessing.load_data()
variable_groups = preprocessing.get_variable_groups(patients)

# Target indexed by admission number, the same index used by the feature matrix
target = preprocessing.create_target(patients)
target.index = patients["inpatient.number"].values

print(f"Internações: {len(target)} | eventos (óbito ou readmissão em 6 meses): {target.sum()} ({target.mean():.1%})")

## 2. Construção da matriz de atributos

**Objetivo**: transformar as 166 colunas originais e a tabela de medicamentos em uma matriz totalmente numérica, sem vazamento de informação.

`features.build_model_matrix()` aplica apenas transformações **determinísticas**, que não dependem do alvo nem aprendem parâmetros dos dados. Por isso podem ser feitas antes da separação treino/teste. O que é aprendido dos dados (imputação, padronização, seleção) fica para dentro do `Pipeline`, ajustado só no treino.

Passos:
1. Junta os 25 fármacos como colunas 0/1 e remove as colunas de vazamento (desfechos, dados da alta, identificador).
2. Remove colunas totalmente vazias.
3. Trata os faltantes de acordo com o seu significado (seção 2.1).
4. Codifica as variáveis qualitativas (seção 2.2).

In [ ]:
X, panel_table = features.build_model_matrix(patients, drugs, variable_groups)

assert X.index.equals(target.index)
assert all(pd.api.types.is_numeric_dtype(X[c]) for c in X.columns)
print(f"\nMatriz de atributos: {X.shape[0]} internações x {X.shape[1]} colunas (todas numéricas)")

### 2.1 Faltantes: três situações diferentes

Um valor vazio nesta base pode significar três coisas, e cada uma recebe um tratamento:

| Situação | Onde ocorre | Tratamento |
|---|---|---|
| **Estrutural**: o vazio é a resposta | `respiratory.support.` — vazio = sem ventilação mecânica | Vira a coluna 0/1 `respiratory.support` (ventilação invasiva e não invasiva juntas, 42 casos) |
| **Informativo**: o exame não foi pedido | Variáveis com mais de 20% de faltantes (gasometria, ecocardiograma, lipídios...) | Uma coluna 0/1 `..._not_ordered` por painel; o valor medido é mantido e imputado depois |
| **Esporádico**: perda ocasional | Variáveis com até 20% de faltantes | Fica como `NaN` e é imputado pela mediana dentro do `Pipeline` |

Exames do mesmo painel faltam juntos, então compartilham um único indicador (variáveis cujos padrões de falta coincidem em pelo menos 98% das internações).

**Como ler a tabela abaixo**: cada linha é um indicador; `n_variables` é quantas variáveis ele cobre e `not_ordered_pct` é a porcentagem de internações em que nenhuma delas foi medida.

In [ ]:
pd.set_option("display.max_colwidth", 90)
panel_table

In [ ]:
# Sanity check of the three situations
indicator_cols = panel_table["indicator"].tolist()
missing_rate = X.isna().mean()

print(f"Estrutural : respiratory.support = 1 em {X['respiratory.support'].sum()} internações")
print(f"Informativo: {len(indicator_cols)} indicadores cobrindo {panel_table['n_variables'].sum()} variáveis")
print(f"Esporádico : {((missing_rate > 0) & (missing_rate <= 0.20)).sum()} variáveis com até 20% de NaN (imputação no Pipeline)")
print(f"Ainda com NaN (a imputar no Pipeline): {(missing_rate > 0).sum()} colunas")

### 2.2 Codificação das variáveis qualitativas

| Tipo | Variáveis | Codificação |
|---|---|---|
| Dois níveis | `gender`, `admission.way`, `type.II.respiratory.failure`, `oxygen.inhalation` | Uma coluna 0/1 |
| Mais de dois níveis | `admission.ward`, `occupation`, `type.of.heart.failure`, `NYHA`, `Killip.grade`, `ageCat` | *One-hot*: uma coluna 0/1 por nível |
| Redundantes | `consciousness`, `eye.opening`, `verbal.response`, `movement` | Removidas: `GCS` é a soma exata dos três componentes e `consciousness` repete a mesma informação |

Detalhes do *one-hot*:
- **Níveis raros são agrupados** antes: `ICU` (15) entra em `Others` no setor de admissão; `worker` (17) e `Officer` (7) entram em `Others` na ocupação; as faixas etárias abaixo de 50 anos viram `(21,49]`.
- **O nível mais frequente é omitido** e serve de referência (ex.: NYHA III). Sem isso as colunas de uma mesma variável somariam sempre 1, o que causa colinearidade perfeita em modelos lineares sem regularização.
- Os 27 pacientes sem ocupação registrada ficam com zero em todas as colunas, ou seja, no nível de referência (`UrbanResident`, que também é a moda).

**Como ler a tabela abaixo**: cada linha é uma coluna criada pela codificação, com o número e a porcentagem de internações em que vale 1.

In [ ]:
encoded_cols = [
    c for c in X.columns
    if c not in patients.columns and not c.startswith("drug_") and c not in indicator_cols
]
encoded_summary = pd.DataFrame({
    "n": X[encoded_cols].sum().astype(int),
    "pct": (X[encoded_cols].mean() * 100).round(1),
})
encoded_summary

In [ ]:
# Composition of the final matrix
drug_cols = [c for c in X.columns if c.startswith("drug_")]
binary_cols = [c for c in X.columns if X[c].dropna().isin([0, 1]).all()]

composition = pd.Series({
    "Numéricas (contínuas ou discretas)": X.shape[1] - len(binary_cols),
    "Binárias clínicas e codificadas": len(binary_cols) - len(drug_cols) - len(indicator_cols),
    "Indicadores de exame não pedido": len(indicator_cols),
    "Fármacos": len(drug_cols),
    "Total": X.shape[1],
}, name="colunas")
composition.to_frame()

## 3. Separação treino/teste

**Objetivo**: reservar um conjunto de teste que nenhuma decisão do trabalho (seleção de atributos, imputação, ajuste de hiperparâmetros) chega a ver. Ele só é usado na avaliação final, para estimar o erro em pacientes novos.

- 80% treino / 20% teste, **estratificado** pelo alvo para manter a mesma proporção de eventos nos dois conjuntos.
- Semente fixa (`RANDOM_STATE`), então a divisão é idêntica em todos os notebooks que chamarem `preprocessing.split_train_test()`.
- A seleção de atributos e a validação cruzada das próximas etapas usam **somente o treino**.

In [ ]:
X_train, X_test, y_train, y_test = preprocessing.split_train_test(X, target)

split_summary = pd.DataFrame({
    "internações": [len(y_train), len(y_test)],
    "eventos": [int(y_train.sum()), int(y_test.sum())],
    "eventos_pct": [round(y_train.mean() * 100, 1), round(y_test.mean() * 100, 1)],
}, index=["treino", "teste"])
split_summary

## 4. Seleção de atributos

Abordagem **híbrida** (recomendada na Aula 06): um filtro simples reduz a redundância e um método embutido (Lasso) faz a seleção propriamente dita. A partir daqui **só o conjunto de treino é usado**.

1. Filtro sem olhar o alvo (4.1)
2. Regressão logística com penalidade L1 em validação cruzada (4.2)
3. Frequência de seleção em reamostragens (4.3)
4. Comparação com um Random Forest (4.4)

### 4.1 Filtro sem olhar o alvo

**Objetivo**: retirar colunas que não podem ajudar nenhum modelo, sem consultar o desfecho (portanto sem risco de sobreajuste).

- **Constantes**: um único valor no treino.
- **Quase constantes**: binárias em que o valor menos comum aparece em menos de 1% das internações (menos de 17 casos no treino).
- **Redundantes**: pares com correlação de Spearman acima de 0,9 em módulo. De cada par fica a variável com menos faltantes. Usamos Spearman em vez de Pearson porque os exames laboratoriais têm distribuições assimétricas e valores extremos (Aula 06).

**Como ler a tabela**: cada linha é uma coluna removida, com o motivo; para as redundantes, `kept_instead` é a variável que ficou no lugar e `abs_spearman` a correlação do par.

In [ ]:
kept_columns, filter_report = features.unsupervised_filter(X_train)
X_train_filtered = X_train[kept_columns]

print(f"Colunas: {X_train.shape[1]} -> {len(kept_columns)}")
print(filter_report["reason"].value_counts().to_string())

filter_report.to_csv(utils.RESULTS_DIR / "feature_filter_report.csv", index=False)
filter_report

### 4.2 Lasso: regressão logística com penalidade L1

**Objetivo**: escolher o subconjunto de atributos que prevê o desfecho em conjunto. A penalidade L1 zera os coeficientes dos atributos que não contribuem, então o próprio ajuste do modelo faz a seleção (Aulas 06 e 07).

**Como é feito**:
- `Pipeline`: imputação pela mediana -> padronização -> regressão logística L1. A padronização é necessária porque a penalidade atua sobre o tamanho dos coeficientes, que depende da escala de cada variável.
- A força da penalidade é controlada por `C` (o inverso de λ): `C` pequeno = penalidade forte = poucos atributos.
- Para cada `C`, validação cruzada estratificada com 10 *folds*. O `Pipeline` inteiro é reajustado em cada *fold* de treino, de modo que a imputação, a padronização e a seleção nunca veem o *fold* de validação.
- Métrica: AUC (área sob a curva ROC).

**Escolha de `C`: regra de um erro padrão** (Aula 07). Em vez do `C` de maior AUC, fica o modelo mais simples cuja AUC está a até um erro padrão do máximo. Vários tamanhos de modelo têm desempenho praticamente igual, e entre eles o menor é o mais interpretável e o menos sujeito a sobreajuste.

In [ ]:
lasso_path = features.lasso_path_cv(X_train_filtered, y_train, features.LASSO_C_GRID)
C_best, C_one_se = features.pick_one_standard_error(lasso_path)

lasso_path.to_csv(utils.RESULTS_DIR / "lasso_path_cv.csv", index=False)
lasso_path.loc[lasso_path["C"].isin([C_best, C_one_se])].round(4)

**Como ler o gráfico**: a curva azul é a AUC média nos 10 *folds* para cada valor de `C`, com barras de ± 1 erro padrão; a curva cinza (eixo da direita) é o número médio de atributos com coeficiente diferente de zero. A linha pontilhada marca o `C` de maior AUC e a tracejada o `C` escolhido pela regra de um erro padrão. À esquerda a penalidade é forte demais (nenhum atributo, AUC 0,5); à direita entram atributos que não melhoram a previsão e a AUC cai.

In [ ]:
fig = features.plot_lasso_path(lasso_path, C_best, C_one_se)
utils.save_fig(fig, "lasso_path_cv")

**Atributos selecionados**: o modelo com o `C` escolhido é reajustado em todo o treino.

**Como ler a tabela**: `coefficient` é o coeficiente na escala padronizada (positivo = aumenta o risco de óbito ou readmissão); `odds_ratio_per_sd` é quanto a chance do evento é multiplicada quando o atributo aumenta um desvio padrão. Os coeficientes do Lasso são encolhidos em direção a zero, então servem para ordenar os atributos, não como estimativa do efeito clínico.

In [ ]:
selected = features.lasso_selected_features(X_train_filtered, y_train, C_one_se)
selected_features = selected.index.tolist()

print(f"{len(selected_features)} atributos selecionados de {len(kept_columns)} (C = {C_one_se:.4f})")
selected.to_csv(utils.RESULTS_DIR / "lasso_selected_features.csv", index_label="feature")
selected

### 4.3 Estabilidade: frequência de seleção em reamostragens

**Objetivo**: saber quais dos atributos escolhidos pelo Lasso são escolhidos de forma consistente. Entre atributos correlacionados o Lasso fica com um deles de modo quase arbitrário, e com sinal fraco a lista pode mudar com pequenas mudanças nos dados.

**Como é feito**: *bootstrap* (Aula 05). O treino é reamostrado 200 vezes com reposição; em cada reamostra o mesmo `Pipeline` do Lasso é reajustado com o `C` escolhido em 4.2 e registramos quais atributos ficaram com coeficiente diferente de zero.

**Como ler a tabela**: `selection_frequency` é a fração das 200 reamostras em que o atributo foi selecionado; `positive_share` é a fração dessas seleções em que o coeficiente foi positivo (1 = sempre aumenta o risco, 0 = sempre reduz; valores intermediários indicariam um efeito de sinal instável); `in_lasso` indica se o atributo está entre os selecionados em 4.2.

In [ ]:
selection_frequency = features.lasso_selection_frequency(X_train_filtered, y_train, C_one_se, n_resamples=200)
selection_frequency["in_lasso"] = selection_frequency.index.isin(selected_features)

for threshold in (0.9, 0.8, 0.6, 0.5):
    print(f"Selecionados em pelo menos {threshold:.0%} das reamostras: {(selection_frequency['selection_frequency'] >= threshold).sum()}")

selection_frequency.to_csv(utils.RESULTS_DIR / "lasso_selection_frequency.csv", index_label="feature")
selection_frequency.head(30)

**Como ler o gráfico**: cada barra é um atributo e o comprimento é a frequência de seleção. Em azul, os atributos selecionados pelo Lasso em 4.2; em cinza, os que ficaram de fora. A linha tracejada marca 50%: à direita dela o atributo é escolhido na maioria das reamostras.

In [ ]:
fig = features.plot_selection_frequency(selection_frequency, selected_features)
utils.save_fig(fig, "lasso_selection_frequency")

### 4.4 Comparação com um Random Forest

**Objetivo**: o Lasso só enxerga efeitos lineares. Esta seção verifica duas coisas: (a) se um modelo não linear dá importância a atributos que o Lasso descartou e (b) se o conjunto escolhido pelo Lasso também serve para um modelo não linear.

**(a) Importância por permutação.** Um Random Forest é treinado com os 168 atributos; em cada *fold* de validação, os valores de um atributo são embaralhados e medimos quanto a AUC cai. Usamos permutação em vez da importância por redução de impureza (Aula 06) porque esta favorece atributos contínuos, com muitos pontos de corte possíveis, e é medida nos próprios dados de treino.

**Como ler a tabela**: `auc_drop_mean` é a queda média de AUC ao embaralhar o atributo (maior = mais importante) e `auc_drop_se` o erro padrão entre os *folds*; `selection_frequency` e `in_lasso` vêm da seção 4.3. Com muitos atributos correlacionados as quedas individuais são pequenas, porque a floresta compensa a perda de um atributo com os outros.

In [ ]:
forest_importance, forest_aucs = features.forest_permutation_importance(X_train_filtered, y_train)
forest_importance = forest_importance.join(selection_frequency[["selection_frequency", "in_lasso"]])

print(f"AUC do Random Forest com todos os atributos (5 folds): {np.mean(forest_aucs):.3f}")
print(f"Entre os 15 mais importantes para a floresta, {int(forest_importance.head(15)['in_lasso'].sum())} estão no conjunto do Lasso")

forest_importance.to_csv(utils.RESULTS_DIR / "forest_permutation_importance.csv", index_label="feature")
forest_importance.head(15)

**(b) O conjunto do Lasso serve para outros modelos? E um filtro simples faria o mesmo?** Comparamos a AUC em validação cruzada (10 *folds*) de dois modelos, um não linear (Random Forest) e um linear (regressão logística sem penalidade), cada um treinado com três conjuntos de atributos:

- **todas**: os 168 atributos após o filtro da seção 4.1 (sem seleção supervisionada);
- **Lasso**: os atributos que o Lasso seleciona (método embutido);
- **filtro F**: o mesmo número de atributos, tirados do topo do *ranking* univariado pelo teste F da ANOVA (método de filtro, Aula 06).

As duas seleções são refeitas dentro de cada *fold* de treino, para que a comparação não seja otimista.

**Como ler a tabela**: uma linha por combinação de modelo e conjunto de atributos; `auc_mean` é a AUC média e `auc_se` o erro padrão. Diferenças menores que um erro padrão não são distinguíveis de ruído.

In [ ]:
set_comparison, mean_size = features.compare_feature_sets_cv(X_train_filtered, y_train, C_one_se)

print(f"Tamanho médio dos conjuntos Lasso e filtro F nos folds: {mean_size:.1f} atributos")
set_comparison.to_csv(utils.RESULTS_DIR / "feature_set_comparison.csv")
set_comparison

### 4.5 Conjunto final de atributos

**Regra**: fica o atributo que foi escolhido pelo Lasso com a regra de um erro padrão (4.2) **e** selecionado em pelo menos 50% das reamostras (4.3). A lista é salva em `results/selected_features.csv` e lida pelos próximos notebooks com `features.load_selected_features()`.

In [ ]:
MIN_SELECTION_FREQUENCY = features.MIN_SELECTION_FREQUENCY  # 0.50

final_features = [
    f for f in selected_features
    if selection_frequency.loc[f, "selection_frequency"] >= MIN_SELECTION_FREQUENCY
]
features.save_selected_features(final_features)

print(f"Conjunto final: {len(final_features)} atributos")
print(f"Retirados por frequência abaixo de {MIN_SELECTION_FREQUENCY:.0%}: {sorted(set(selected_features) - set(final_features))}")
selection_frequency.loc[final_features, ["selection_frequency"]].join(selected["coefficient"])

**Conferência com critérios que penalizam o número de atributos.** A escolha acima usou a AUC em validação cruzada, que estima diretamente o erro de teste: como cada *fold* é avaliado em dados que o modelo não viu, acrescentar atributos inúteis não melhora o resultado (na seção 4.4 a regressão logística com os 168 atributos é pior do que com os selecionados). A Aula 07 apresenta a alternativa indireta: corrigir o erro de treino com uma penalidade pelo tamanho do modelo (Cp, AIC, BIC, R² ajustado). O R² ajustado e o Cp são definidos para regressão por mínimos quadrados; para a regressão logística os equivalentes são o AIC e o BIC, calculados a partir da *deviance*.

**Como ler a tabela**: cada linha é um conjunto candidato, ajustado por regressão logística sem penalidade no treino. `deviance` mede o erro de treino e sempre cai quando entram atributos; `AIC` soma 2 por coeficiente e `BIC` soma log(n) ≈ 7,4 por coeficiente. **Menor é melhor.** Como os conjuntos foram escolhidos nestes mesmos dados, os valores servem para comparar tamanhos, não como estimativa do erro.

In [ ]:
top_forest_extra = ["high.sensitivity.troponin", "D.dimer", "albumin"]
candidate_sets = {
    "Frequência >= 60%": [f for f in selected_features if selection_frequency.loc[f, "selection_frequency"] >= 0.60],
    "Final (Lasso 1 EP e frequência >= 50%)": final_features,
    "Lasso, regra de 1 erro padrão": selected_features,
    "Final + destaques do Random Forest": final_features + top_forest_extra,
    "Lasso, maior AUC": features.lasso_selected_features(X_train_filtered, y_train, C_best).index.tolist(),
    "Todos após o filtro": kept_columns,
}
criteria = features.information_criteria(X_train_filtered, y_train, candidate_sets)
criteria.to_csv(utils.RESULTS_DIR / "feature_set_information_criteria.csv", index_label="feature_set")
criteria